In [1]:
import pandas as pd
from src_RF_DT import *
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df_train = pd.read_parquet("enem_parquet", columns = colunas)
df_test = pd.read_parquet("2023", columns = colunas)

In [3]:
df_train = pre_processor_rf_dt(df_train, objetivo = '', n_samples = 100_000)
df_test = pre_processor_rf_dt(df_test, objetivo = '', n_samples = 100_000)

In [4]:
X_train = df_train.drop(['FALTOU'], axis=1)
y_train = df_train['FALTOU']

X_test = df_test.drop(['FALTOU'], axis=1)
y_test = df_test['FALTOU']

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [6]:
clf = RandomForestClassifier()

clf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, clf.predict(X_train))))
print('Eval: %0.4f' % (1 - accuracy_score(y_val,  clf.predict(X_val))))

print(classification_report(y_val, clf.predict(X_val)))

Ein:  0.0176
Eval: 0.3421
              precision    recall  f1-score   support

           0       0.70      0.77      0.74     12243
           1       0.57      0.47      0.51      7652

    accuracy                           0.66     19895
   macro avg       0.63      0.62      0.63     19895
weighted avg       0.65      0.66      0.65     19895



In [ ]:
cv_rf = tune_random_forest(X_train, y_train, n_iter=30, cv=5, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomForestClassifier(class_weight='balanced', max_features='log2',
                       min_samples_leaf=10, min_samples_split=20,
                       n_estimators=166, random_state=42)
Ein:  0.2832
Eout: 0.3330
              precision    recall  f1-score   support

           0       0.76      0.67      0.71     12243
           1       0.56      0.67      0.61      7652

    accuracy                           0.67     19895
   macro avg       0.66      0.67      0.66     19895
weighted avg       0.68      0.67      0.67     19895



In [11]:
rf = RandomForestClassifier(
    **cv_rf.best_params_,
    class_weight='balanced',  
    random_state=42
)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

#joblib.dump(rf, 'rf_presenca.pkl')

Ein:  0.2832
Eout: 0.3341
              precision    recall  f1-score   support

           0       0.78      0.70      0.74     68079
           1       0.48      0.58      0.53     31921

    accuracy                           0.67    100000
   macro avg       0.63      0.64      0.63    100000
weighted avg       0.69      0.67      0.67    100000

